# plot features for Nana
*by **David W. Hogg** (NYU).*

Execute this notebook and then read the commentary below for Hogg's conclusions.

Before you execute this notebook, be sure to move the file `good_parentsfit.fits` into the `data` subdirectory of this repo.

In [ ]:
# !pip install matplotlib
# !pip install astropy

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits

In [ ]:
# read in Nana's data
fn = "../data/good_parentsfit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
# turn the real features into the complex numbers they so richly deserve to be
n = len(data)
m = 32 # hard coded
complex_features = (data["features"])[:, 1::2] + 1j * (data["features"])[:, 2::2]
print(complex_features[13])

In [ ]:
# make real scalars, which are essentially powers in a discrete power spectrum
scalars = (complex_features * complex_features.conj()).real
variances = np.sum(scalars, axis=1)
print(scalars[13])
print(len(data), scalars.shape, variances.shape)

In [ ]:
# exclude very long and very short refined frequencies, because these are not real (we think)
good = (variances > 1.e-5) \
     * (data["refined_frequency"] < 24) \
     * (data["refined_frequency"] > 0.01)

In [ ]:
# naively, the highest "information" features these should be the most valuable for us going forward
informations = variances * data["refined_frequency"]

In [ ]:
# plot a few plots of the features to show that we have structure in the data
# symbol sizes are (log) informations, colors are (log) frequencies
sizes = 0.5 * np.log10(informations)
sizes -= np.nanmin(sizes)
sizes += 0.01
for i, j in [(0, 1),
             (0, 2),
             (0, 3),
             (1, 2)]:
    plt.axhline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.axvline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.scatter(scalars[good, i], scalars[good, j], c=np.log10(data[good]["refined_frequency"]), s=sizes[good])
    plt.loglog()
    plt.xlabel(f"scalar {i}")
    plt.ylabel(f"scalar {j}")
    plt.colorbar(label="log_10 frequency")
    plt.savefig(f"scatter_{i}_{j}.png")
    plt.show()

## What do we conclude from the above?
I (Hogg) believe that these plots show that there is abundant information in the features to do very interesting machine-learning tasks, including classifications.

In [ ]:
# now choose a few "very best" objects
verygood = (informations > 30.0) * good
print(np.sum(verygood))

In [ ]:
# plot their 
foo, k = scalars.shape
for i in np.arange(len(data)):
    if verygood[i]:
        plt.axhline(1.0, color="k", lw=1.0, alpha=0.5)
        plt.scatter(data[i]["refined_frequency"] * np.arange(k), scalars[i], c="k")
        plt.title(data[i]["star_id"])
        plt.xlabel("frequency (inverse days)")
        plt.ylabel("power")
        plt.semilogy()
        plt.show()

## What do we conclude from the above?
First of all, note that a power (a scalar) can never get larger than 1. Why not? Because the data are normalized to unity; if a scalar gets larger than unity, the lightcurve would be going negative.

Note that many powers *do* get larger than 1. Always at or near resonances with the data sampling frequency at 48-ish. Why? Because near these frequencies, the *observed* variation from a large amplitude is small: The integration (averaging) over the exposure time *T* nulls the signal.

We should be classifying these power spectra using not the square of the amplitude but rather the observed variance in the actual data. This will require importing the design matrix code from Nana's codebase.